# PCOS Synthetic Data Generation (TVAE)


In [4]:
%pip install -q sdv pandas numpy scikit-learn matplotlib

Note: you may need to restart the kernel to use updated packages.


In [1]:
# ============================================================
# PART 1: leak-free split + TVAE synthetic generation (ONE RUN)
# pip install ctgan pandas numpy scikit-learn scipy torch
# ============================================================
import warnings
warnings.filterwarnings("ignore")

import os, json, hashlib, platform, pickle
import numpy as np
import pandas as pd
import torch
import sklearn, scipy, ctgan
from scipy.stats import ks_2samp
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from ctgan import TVAE

try:
    from IPython.display import display
except ImportError:
    display = print

# ---------------- CONFIG ----------------
DATA_PATH = "pcos_clean.csv"
OUT_DIR = "run_final"
TARGET = "PCOS (Y/N)"
CYCLE_COL = "Cycle(R/I)"
CYCLE_MAP = {2: 0, 4: 1}          # 0 = Regular, 1 = Irregular  (CONFIRM!)
NUM_SYNTHETIC_ROWS = 1500
EPOCHS = 500
TEST_SIZE = 0.20
SEED = 42

CATEGORICAL_COLUMNS = [
    "PCOS (Y/N)", "Blood Group", "Cycle(R/I)", "Pregnant(Y/N)",
    "Weight gain(Y/N)", "hair growth(Y/N)", "Skin darkening (Y/N)",
    "Hair loss(Y/N)", "Pimples(Y/N)", "Fast food (Y/N)", "Reg.Exercise(Y/N)"
]

os.makedirs(OUT_DIR, exist_ok=True)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ---------------- HELPERS ----------------
def sha256(path):
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()


def train_tvae(frame):
    model = TVAE(epochs=EPOCHS)
    model.fit(frame, discrete_columns=CATEGORICAL_COLUMNS)
    return model


def postprocess(synth, real):
    out = synth.copy()
    for col in real.columns:
        if col in CATEGORICAL_COLUMNS:
            valid = np.sort(real[col].unique())
            vals = out[col].astype(float).values
            idx = np.abs(vals[:, None] - valid[None, :].astype(float)).argmin(axis=1)
            out[col] = valid[idx]
        else:
            out[col] = out[col].astype(float).clip(real[col].min(), real[col].max())
            if (real[col] % 1 == 0).all():
                out[col] = out[col].round(0)
        out[col] = out[col].astype(real[col].dtype)
    return out[real.columns]


def row_keys(frame):
    return frame.round(6).astype(str).apply(tuple, axis=1)


def sample_matched(model, n_rows, train_frame, forbidden_frames):
    """n_rows synthetic rows.
    - positive rate == train positive rate
    - no duplicates inside synthetic
    - no exact copy of any row in forbidden_frames (train AND test)"""
    rate = train_frame[TARGET].mean()
    n_pos = int(round(rate * n_rows))
    n_neg = n_rows - n_pos

    forbidden = set()
    for f in forbidden_frames:
        forbidden |= set(row_keys(f))

    pool = pd.DataFrame(columns=train_frame.columns)
    for _ in range(100):
        batch = postprocess(model.sample(n_rows * 2), train_frame)
        batch = batch[~row_keys(batch).isin(forbidden)]
        pool = pd.concat([pool, batch], ignore_index=True).drop_duplicates()
        pos = pool[pool[TARGET] == 1]
        neg = pool[pool[TARGET] == 0]
        if len(pos) >= n_pos and len(neg) >= n_neg:
            break
    else:
        raise RuntimeError("Could not collect enough rows for target class ratio")

    out = pd.concat([pos.iloc[:n_pos], neg.iloc[:n_neg]], ignore_index=True)
    out = out.sample(frac=1, random_state=SEED).reset_index(drop=True)
    return out.astype(train_frame.dtypes.to_dict())


def quality_eval(real, synth, label=""):
    num_cols = [c for c in real.columns if c not in CATEGORICAL_COLUMNS]
    scores = {}
    for c in num_cols:
        scores[c] = 1 - ks_2samp(real[c], synth[c]).statistic
    for c in CATEGORICAL_COLUMNS:
        p = real[c].value_counts(normalize=True)
        q = synth[c].value_counts(normalize=True)
        idx = p.index.union(q.index)
        tvd = 0.5 * np.abs(p.reindex(idx, fill_value=0) - q.reindex(idx, fill_value=0)).sum()
        scores[c] = 1 - tvd
    shape_df = pd.DataFrame({"Column": list(scores), "Shape Score": list(scores.values())}
                            ).sort_values("Shape Score")
    col_shape = shape_df["Shape Score"].mean()
    corr_diff = (real[num_cols].corr() - synth[num_cols].corr()).abs().values
    pair_trend = 1 - np.nanmean(corr_diff) / 2
    print(f"\n===== SYNTHETIC QUALITY {label} =====")
    print(f"Column Shape Score     : {col_shape:.4f}")
    print(f"Column Pair Trend Score: {pair_trend:.4f}")
    print(f"Overall Quality Score  : {(col_shape + pair_trend) / 2:.4f}")
    print(f"Positive rate real {real[TARGET].mean():.4f} | synth {synth[TARGET].mean():.4f}")
    print("\nWorst 10 columns:")
    display(shape_df.head(10))
    shape_df.to_csv(os.path.join(OUT_DIR, "quality_column_shape.csv"), index=False)
    return {"col_shape": float(col_shape), "pair_trend": float(pair_trend),
            "overall": float((col_shape + pair_trend) / 2)}


def dcr_report(synth, ref, name):
    sc = StandardScaler().fit(ref)
    nn = NearestNeighbors(n_neighbors=1).fit(sc.transform(ref))
    d = nn.kneighbors(sc.transform(synth))[0].ravel()
    print(f"DCR to {name}: min {d.min():.4f} | median {np.median(d):.4f} | "
          f"mean {d.mean():.4f} | exact copies {int((d == 0).sum())}")
    return d


# ---------------- 1. LOAD + FIX CYCLE ENCODING ----------------
df = pd.read_csv(DATA_PATH)
print("Original shape:", df.shape)

assert set(df[CYCLE_COL].unique()) <= set(CYCLE_MAP), \
    f"Unexpected cycle values: {df[CYCLE_COL].unique()}"
df[CYCLE_COL] = df[CYCLE_COL].map(CYCLE_MAP).astype(int)
print("Cycle values now:", sorted(df[CYCLE_COL].unique()), "(0=Regular, 1=Irregular)")
print("\nTarget distribution:\n", df[TARGET].value_counts())

# ---------------- 2. SPLIT FIRST ----------------
train_df, test_df = train_test_split(
    df, test_size=TEST_SIZE, stratify=df[TARGET], random_state=SEED
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

train_df.to_csv(os.path.join(OUT_DIR, "train_real.csv"), index=False)
test_df.to_csv(os.path.join(OUT_DIR, "test_real.csv"), index=False)
print(f"\nTrain real: {train_df.shape} | pos rate {train_df[TARGET].mean():.4f}")
print(f"Test  real: {test_df.shape} | pos rate {test_df[TARGET].mean():.4f}")

# ---------------- 3. TVAE ON TRAIN ONLY ----------------
model = train_tvae(train_df)
print("TVAE training done (train split only).")

synthetic_df = sample_matched(
    model, NUM_SYNTHETIC_ROWS, train_df, forbidden_frames=[train_df, test_df]
)

# ---------------- 4. QUALITY + PRIVACY ----------------
q = quality_eval(train_df, synthetic_df, label="(TRAIN vs SYNTHETIC)")
print("\n--- Privacy (distance to closest real record) ---")
dcr_train = dcr_report(synthetic_df, train_df, "train")
dcr_test = dcr_report(synthetic_df, test_df, "test (sanity)")

# ---------------- 5. SAVE (one run, together) ----------------
augmented_train = pd.concat([train_df, synthetic_df], ignore_index=True)
augmented_train = augmented_train.sample(frac=1, random_state=SEED).reset_index(drop=True)

syn_path = os.path.join(OUT_DIR, "synthetic_pcos.csv")
aug_path = os.path.join(OUT_DIR, "augmented_train.csv")
synthetic_df.to_csv(syn_path, index=False)
augmented_train.to_csv(aug_path, index=False)

try:
    model.save(os.path.join(OUT_DIR, "tvae_model.pt"))
except Exception:
    with open(os.path.join(OUT_DIR, "tvae_model.pkl"), "wb") as f:
        pickle.dump(model, f)

# ---------------- 6. LEAK CHECKS (must all pass) ----------------
train_keys, test_keys = set(row_keys(train_df)), set(row_keys(test_df))
syn_keys = set(row_keys(synthetic_df))
assert len(synthetic_df) == NUM_SYNTHETIC_ROWS
assert len(syn_keys) == len(synthetic_df), "duplicates inside synthetic"
assert not (syn_keys & test_keys), "synthetic copies a TEST row"
assert not (syn_keys & train_keys), "synthetic copies a TRAIN row"
assert not (train_keys & test_keys), "train/test overlap"
assert len(augmented_train) == len(train_df) + NUM_SYNTHETIC_ROWS
print("\nAll leak checks passed.")

# ---------------- 7. MANIFEST ----------------
manifest = {
    "seed": SEED, "epochs": EPOCHS, "test_size": TEST_SIZE,
    "num_synthetic_rows": NUM_SYNTHETIC_ROWS,
    "cycle_map": {str(k): v for k, v in CYCLE_MAP.items()},
    "n_real": len(df), "n_train": len(train_df), "n_test": len(test_df),
    "train_pos_rate": float(train_df[TARGET].mean()),
    "test_pos_rate": float(test_df[TARGET].mean()),
    "synthetic_pos_rate": float(synthetic_df[TARGET].mean()),
    "quality": q,
    "dcr_train_min": float(dcr_train.min()),
    "dcr_train_median": float(np.median(dcr_train)),
    "versions": {"python": platform.python_version(), "numpy": np.__version__,
                 "pandas": pd.__version__, "sklearn": sklearn.__version__,
                 "scipy": scipy.__version__, "torch": torch.__version__,
                 "ctgan": ctgan.__version__},
    "sha256": {"pcos_clean.csv": sha256(DATA_PATH),
               "train_real.csv": sha256(os.path.join(OUT_DIR, "train_real.csv")),
               "test_real.csv": sha256(os.path.join(OUT_DIR, "test_real.csv")),
               "synthetic_pcos.csv": sha256(syn_path),
               "augmented_train.csv": sha256(aug_path)},
}
with open(os.path.join(OUT_DIR, "run_manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2)

print("\nSaved to", OUT_DIR, ":", sorted(os.listdir(OUT_DIR)))

Original shape: (541, 42)


AssertionError: Unexpected cycle values: [2 4 5]